In [ ]:
# Setup for Google Colab
import sys, os
if 'google.colab' in sys.modules:
    # Downgrade ipywidgets for nglview compatibility
    !pip install -q mrcfile gemmi nglview ipywidgets==7.7.1
    if not os.path.exists('/content/synth-cryo-em'):
        !git clone https://github.com/elkins/synth-cryo-em.git /content/synth-cryo-em
    sys.path.append('/content/synth-cryo-em/src')
    os.chdir('/content/synth-cryo-em')
    # Enable third-party widgets in Colab
    from google.colab import output
    output.enable_custom_widget_manager()


> **Note for Google Colab Users**: 
> When you run the setup cell above, Colab may display a blue bar or a popup asking to **'Enable third-party Jupyter widgets'**. 
> Please click **'Enable'** to allow the 3D visualization (nglview) to render correctly.

# Interactive Cryo-EM Simulation: Physics & Visualization

This tutorial demonstrates how to use `synth-cryo-em` to generate synthetic maps and visualize them interactively. 

We will explore:
1. **Map-to-Model Fit**: Overlaying the atomic model with its synthetic density.
2. **The Physics of Resolution**: How resolution, noise, and CTF effects change the visual features of a map.

In [ ]:
import os
import numpy as np
import nglview as nv
import ipywidgets as widgets
from IPython.display import display
from synth_cryo_em.core import generate_density_map, apply_ctf, add_gaussian_noise, save_mrc

## 1. Prepare the Data
We'll use a small standard PDB for this demonstration.

In [ ]:
import os
# Use sample.pdb which contains actual atomic coordinates
if 'google.colab' in sys.modules:
    pdb_path = '/content/synth-cryo-em/tests/sample.pdb'
else:
    pdb_path = 'tests/sample.pdb'

if not os.path.exists(pdb_path):
    # Fallback for local run from different directory
    if os.path.exists('../tests/sample.pdb'):
        pdb_path = '../tests/sample.pdb'

print(f"Using model: {os.path.abspath(pdb_path)}")
if not os.path.exists(pdb_path):
    raise FileNotFoundError(f"Critical Error: PDB file not found at {pdb_path}")

## 2. Interactive Simulation Dashboard
Use the sliders below to adjust the physical parameters. The map will be re-generated and visualized automatically.

In [ ]:
def update_visuals(resolution, snr, defocus, apply_physics):
    global pdb_path
    print(f"[DEBUG] update_visuals called with pdb_path: {pdb_path}")
    if not os.path.exists(pdb_path):
        print(f"[DEBUG] ERROR: {pdb_path} does not exist!")
    # 1. Generate core density
    grid, origin = generate_density_map(pdb_path, resolution=resolution)
    data = np.array(grid, copy=True)
    
    # 2. Apply Physics
    if apply_physics:
        uc = grid.unit_cell
        vox_size = (uc.a / grid.nu, uc.b / grid.nv, uc.c / grid.nw)
        data = apply_ctf(data, vox_size, defoc=defocus)
    
    # 3. Add Noise
    if snr < 50: # Use 50 as 'clean' threshold
        data = add_gaussian_noise(data, snr=snr)
        
    # Save temporary MRC for nglview
    temp_mrc = "temp_viz.mrc"
    uc = grid.unit_cell
    spacing = (uc.a / grid.nu, uc.b / grid.nv, uc.c / grid.nw)
    save_mrc(data, temp_mrc, origin=origin, spacing=spacing)
    
    # Visualization
    view = nv.show_file(pdb_path)
    view.add_component(temp_mrc)
    
    # Adjust map representation
    view.component_1.clear_representations()
    view.component_1.add_surface(isolevel=np.max(data)*0.2, 
                                 opacity=0.4, 
                                 color='skyblue')
    
    display(view)
    print(f"Current State: Res={resolution}A, SNR={snr}, Defocus={defocus}um")
# Widgets
res_slider = widgets.FloatSlider(value=4.0, min=2.0, max=10.0, step=0.5, description='Resolution (A)')
snr_slider = widgets.FloatSlider(value=50.0, min=1.0, max=50.0, step=1.0, description='SNR')
defocus_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Defocus (um)')
physics_toggle = widgets.Checkbox(value=False, description='Apply CTF')
ui = widgets.VBox([res_slider, snr_slider, defocus_slider, physics_toggle])
out = widgets.interactive_output(update_visuals, {
    'resolution': res_slider, 
    'snr': snr_slider, 
    'defocus': defocus_slider,
    'apply_physics': physics_toggle
})
display(ui, out)
